##Transform Results Data

1. Read bronze results table
2. Keep only the columns required for analytics (Drop url column)
3. Standardise column names using snake_case (constructorId->constructor_id, driverId->driver_id, raceName->race_name, positionText->finish_position_text)
4. Rename columns to make them more meaningful (date->race_date, grid->grid_position, laps->completed_laps, number->car_number, position->finish_position)
5. Filter out rows where season, round, constructor_id or driver_id is null (business key validation)
6. Remove duplicate records
7. Transform values of columns race_name to Title Case
8. Write the transformed data to silver results table

In [0]:
%run ../00-common/01.environment-config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.results"
silver_table = f"{catalog_name}.{silver_schema}.results"

In [0]:
from pyspark.sql import functions as F

####Step 1 to 7: Read, Transform, perform quality checks

In [0]:
results_df = (
    spark.table(bronze_table)
        .select("season",
                "round",
                "constructorId",
                "driverId",
                "date",
                "raceName",
                "grid",
                "laps",
                "number",
                "points",
                "position",
                "positionText",
                "status",
                "ingestion_timestamp",
                "source_file"
            )
        .withColumnsRenamed({
                "constructorId": "constructor_id",
                "driverId": "driver_id",
                "raceName": "race_name",
                "date": "race_date",
                "grid": "grid_position",
                "laps": "completed_laps",
                "number": "driver_number",
                "position": "final_position",
                "positionText": "final_position_text"})
        .filter(
                F.col("season").isNotNull() &
                F.col("round").isNotNull() &
                F.col("constructor_id").isNotNull() &
                F.col("driver_id").isNotNull()
            )
        .dropDuplicates(["season", "round", "constructor_id", "driver_id"])
        .withColumn("race_name", F.initcap(F.col("race_name")))

)

####Sometimes the fully chained method becomes overwhelming and hence developers use a middle-ground approach where they only chain similar steps like selection + renaming, filtering + removing duplicates, etc

####Step 8: Write the transformed data to the silver 'results' table

In [0]:
(
    results_df
        .write
        .mode("overwrite")
        .format("delta")
        .saveAsTable(silver_table)
)

In [0]:
display(spark.table(silver_table))

season,round,constructor_id,driver_id,race_date,race_name,grid_position,completed_laps,driver_number,points,final_position,final_position_text,status,ingestion_timestamp,source_file
1950,1,alfa,farina,1950-05-13,British Grand Prix,1,70,2,9.0,1,1,Finished,2026-07-06T16:17:15.603Z,dbfs:/Volumes/formula1/landing/files/results/results_1950.json
1950,3,wetteroth,rathmann,1950-05-30,Indianapolis 500,28,122,76,0.0,24,24,+16 Laps,2026-07-06T16:17:15.603Z,dbfs:/Volumes/formula1/landing/files/results/results_1950.json
1950,3,kurtis_kraft,agabashian,1950-05-30,Indianapolis 500,2,64,28,0.0,28,R,Oil leak,2026-07-06T16:17:15.603Z,dbfs:/Volumes/formula1/landing/files/results/results_1950.json
1950,4,maserati,bira,1950-06-04,Swiss Grand Prix,8,40,30,3.0,4,4,+2 Laps,2026-07-06T16:17:15.603Z,dbfs:/Volumes/formula1/landing/files/results/results_1950.json
1950,5,maserati,branca,1950-06-18,Belgian Grand Prix,11,29,30,0.0,10,10,+6 Laps,2026-07-06T16:17:15.603Z,dbfs:/Volumes/formula1/landing/files/results/results_1950.json
1950,6,lago,pozzi,1950-07-02,French Grand Prix,15,56,26,0.0,6,6,+8 Laps,2026-07-06T16:17:15.603Z,dbfs:/Volumes/formula1/landing/files/results/results_1950.json
1950,7,alfa,taruffi,1950-09-03,Italian Grand Prix,7,34,60,0.0,13,R,Engine,2026-07-06T16:17:15.603Z,dbfs:/Volumes/formula1/landing/files/results/results_1950.json
1951,1,ferrari,taruffi,1951-05-27,Swiss Grand Prix,6,42,44,6.0,2,2,Finished,2026-07-06T16:17:15.603Z,dbfs:/Volumes/formula1/landing/files/results/results_1951.json
1951,1,alfa,sanesi,1951-05-27,Swiss Grand Prix,4,41,28,3.0,4,4,+1 Lap,2026-07-06T16:17:15.603Z,dbfs:/Volumes/formula1/landing/files/results/results_1951.json
1951,1,lago,etancelin,1951-05-27,Swiss Grand Prix,12,39,4,0.0,10,10,+3 Laps,2026-07-06T16:17:15.603Z,dbfs:/Volumes/formula1/landing/files/results/results_1951.json
